# Ingredient Recognition using Roboflow Inference SDK

**Purpose**: Detect and identify raw food ingredients from uploaded photos using Roboflow serverless inference

**Task**: T014-T017 [US1] (Consolidated)

**Model**: food-ingredients-dataset/2 (Roboflow)

**Outputs**:
- Ingredient detection with bounding boxes
- Confidence scores and classifications
- Size estimation from bounding boxes
- Ingredient entity creation per data-model.md

## 1. Environment Setup

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from pathlib import Path
from PIL import Image
import json
from datetime import datetime
import uuid

# Roboflow Inference SDK
from inference_sdk import InferenceHTTPClient

# Set random seed
np.random.seed(42)

print("✅ Packages imported successfully")

## 2. Load Model Configuration

In [ ]:
# Project paths
PROJECT_ROOT = Path.cwd().parent.parent
MODEL_DIR = PROJECT_ROOT / "models" / "ingredient_recognition"
DATA_TEST = PROJECT_ROOT / "data" / "test_images"
RESULTS_DIR = PROJECT_ROOT / "data" / "results" / "ingredient_detection"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Roboflow configuration
ROBOFLOW_API_KEY = os.environ.get('ROBOFLOW_API_KEY', 'kuzgSqDiqDLJkXrzcqRr')
API_URL = "https://serverless.roboflow.com"
MODEL_ID = "food-ingredients-dataset/2"
CONFIDENCE_THRESHOLD = 0.7

print(f"📁 Results directory: {RESULTS_DIR}")
print(f"🎯 Model: {MODEL_ID}")
print(f"🔑 API Key: {ROBOFLOW_API_KEY[:10]}...")
print(f"📊 Confidence threshold: {CONFIDENCE_THRESHOLD}")

## 3. Initialize Roboflow Client

In [ ]:
# Initialize Roboflow Inference Client
CLIENT = InferenceHTTPClient(
    api_url=API_URL,
    api_key=ROBOFLOW_API_KEY
)

print("✅ Roboflow Inference Client initialized and ready")

## 4. Ingredient Detection Function

In [ ]:
def detect_ingredients(image_path):
    """
    Detect ingredients in an image using Roboflow
    
    Args:
        image_path: Path to image file
    
    Returns:
        dict: Detection results with predictions
    """
    result = CLIENT.infer(str(image_path), model_id=MODEL_ID)
    return result

print("✅ Detection function defined")

## 5. Size Estimation from Bounding Box

In [ ]:
def estimate_ingredient_size(prediction, image_width, image_height, ingredient_class):
    """
    Estimate ingredient size and weight from bounding box
    
    Args:
        prediction: Single prediction dict from Roboflow
        image_width: Original image width
        image_height: Original image height
        ingredient_class: Ingredient class name
    
    Returns:
        dict: Size estimation with weight, confidence
    """
    # Extract bounding box
    bbox_width = prediction.get('width', 0)
    bbox_height = prediction.get('height', 0)
    
    # Calculate box area ratio
    box_area = bbox_width * bbox_height
    image_area = image_width * image_height
    area_ratio = box_area / image_area if image_area > 0 else 0
    
    # Average weights database (grams) for common ingredients
    average_weights = {
        'chicken': 200,
        'chicken breast': 200,
        'beef': 250,
        'pork': 200,
        'salmon': 150,
        'fish': 150,
        'shrimp': 100,
        'egg': 50,
        'potato': 150,
        'tomato': 100,
        'carrot': 80,
        'broccoli': 100,
        'onion': 120,
        'garlic': 10,
        'apple': 180,
        'banana': 120,
        'orange': 150,
    }
    
    # Get base weight
    ingredient_lower = ingredient_class.lower()
    base_weight = average_weights.get(ingredient_lower, 150)  # Default 150g
    
    # Estimate weight based on area ratio
    # Assume standard photo has ingredient taking 40% of image
    standard_ratio = 0.4
    estimated_weight = base_weight * (area_ratio / standard_ratio)
    
    # Clamp to reasonable range
    min_weight = base_weight * 0.2
    max_weight = base_weight * 5.0
    estimated_weight = np.clip(estimated_weight, min_weight, max_weight)
    
    # Estimate confidence based on area ratio
    if 0.15 < area_ratio < 0.7:
        size_confidence = 0.8
    elif 0.1 < area_ratio < 0.9:
        size_confidence = 0.6
    else:
        size_confidence = 0.4
    
    return {
        'estimated_weight_grams': float(estimated_weight),
        'size_confidence': size_confidence,
        'area_ratio': float(area_ratio),
        'bbox_size': {'width': bbox_width, 'height': bbox_height},
        'estimation_method': 'bounding_box_area'
    }

print("✅ Size estimation function defined")

## 6. Create Ingredient Entity (per data-model.md)

In [ ]:
# Ingredient category mapping
CATEGORY_MAP = {
    # Proteins
    'chicken': 'protein',
    'chicken breast': 'protein',
    'beef': 'protein',
    'pork': 'protein',
    'salmon': 'protein',
    'fish': 'protein',
    'shrimp': 'protein',
    'prawn': 'protein',
    'egg': 'protein',
    'tofu': 'protein',
    
    # Vegetables
    'beetroot': 'vegetable',
    'carrot': 'vegetable',
    'broccoli': 'vegetable',
    'tomato': 'vegetable',
    'potato': 'vegetable',
    'onion': 'vegetable',
    'garlic': 'vegetable',
    'lettuce': 'vegetable',
    'cabbage': 'vegetable',
    'spinach': 'vegetable',
    'pepper': 'vegetable',
    'cucumber': 'vegetable',
    
    # Fruits
    'apple': 'fruit',
    'banana': 'fruit',
    'orange': 'fruit',
    'lemon': 'fruit',
    'strawberry': 'fruit',
    'mango': 'fruit',
    
    # Grains
    'rice': 'grain',
    'pasta': 'grain',
    'noodle': 'grain',
    'bread': 'grain',
    
    # Dairy
    'cheese': 'dairy',
    'milk': 'dairy',
    'butter': 'dairy',
    'yogurt': 'dairy',
}

def get_ingredient_category(ingredient_name):
    """
    Get category for an ingredient
    
    Args:
        ingredient_name: Ingredient name (e.g., 'Beetroot', 'chicken breast')
    
    Returns:
        str: Category (protein/vegetable/fruit/grain/dairy/other)
    """
    # Normalize name
    normalized = ingredient_name.lower().strip()
    
    # Try exact match first
    if normalized in CATEGORY_MAP:
        return CATEGORY_MAP[normalized]
    
    # Try partial match (e.g., 'chicken' in 'chicken breast')
    for key, category in CATEGORY_MAP.items():
        if key in normalized or normalized in key:
            return category
    
    # Default to 'other'
    return 'other'

def create_ingredient_entity(prediction, size_estimation, image_path):
    """
    Create Ingredient entity following data-model.md schema
    
    Args:
        prediction: Roboflow prediction dict
        size_estimation: Size estimation dict
        image_path: Source image path
    
    Returns:
        dict: Ingredient entity
    """
    ingredient_id = str(uuid.uuid4())
    ingredient_name = prediction['class']
    
    entity = {
        # Core fields
        'ingredient_id': ingredient_id,
        'name': ingredient_name,
        'category': get_ingredient_category(ingredient_name),
        'confidence_score': prediction['confidence'],
        
        # Visual characteristics
        'visual_characteristics': {
            'bounding_box': {
                'x': prediction['x'],
                'y': prediction['y'],
                'width': prediction['width'],
                'height': prediction['height']
            },
            'detection_class': prediction['class'],
            'area_ratio': size_estimation['area_ratio']
        },
        
        # Estimated quantity
        'estimated_quantity': {
            'weight_grams': size_estimation['estimated_weight_grams'],
            'confidence': size_estimation['size_confidence'],
            'method': size_estimation['estimation_method']
        },
        
        # Metadata
        'source_image': str(image_path),
        'detected_at': datetime.now().isoformat(),
        'model_version': MODEL_ID
    }
    
    return entity

print("✅ Category mapping defined")
print("✅ Entity creation function defined")
print(f"\n💡 Supported categories: {set(CATEGORY_MAP.values())}")
print(f"   Total mapped ingredients: {len(CATEGORY_MAP)}")

## 7. Validation and Error Handling

In [ ]:
def validate_detection(predictions, min_confidence=0.7):
    """
    Validate detection results per ingredient_recognition.json contract
    
    Args:
        predictions: Detection results
        min_confidence: Minimum confidence threshold
    
    Returns:
        tuple: (is_valid, error_message)
    """
    # Check if any predictions exist
    if 'predictions' not in predictions or len(predictions['predictions']) == 0:
        return False, "No ingredient detected in the image. Please ensure the photo clearly shows a single main ingredient."
    
    # Get highest confidence prediction
    top_prediction = max(predictions['predictions'], key=lambda p: p['confidence'])
    
    # Check confidence threshold
    if top_prediction['confidence'] < min_confidence:
        return False, f"Low confidence detection ({top_prediction['confidence']:.2f}). Please take a clearer photo with better lighting."
    
    return True, None

def handle_detection_error(image_path, error_message):
    """
    Handle detection errors with user-friendly messages
    
    Args:
        image_path: Path to image
        error_message: Error description
    
    Returns:
        dict: Error response
    """
    return {
        'success': False,
        'error': error_message,
        'suggestions': [
            'Ensure the ingredient is well-lit',
            'Place the ingredient on a plain background',
            'Make sure the ingredient fills most of the frame',
            'Avoid blurry or dark images'
        ],
        'image_path': str(image_path)
    }

print("✅ Validation functions defined")

## 8. Complete Ingredient Recognition Pipeline

In [ ]:
def recognize_ingredient(image_path, confidence_threshold=0.7):
    """
    Complete pipeline: detect ingredient, estimate size, create entity
    
    Args:
        image_path: Path to image file
        confidence_threshold: Minimum confidence
    
    Returns:
        dict: Result with ingredient entity or error
    """
    try:
        # 1. Validate image exists
        if not Path(image_path).exists():
            return handle_detection_error(image_path, "Image file not found")
        
        # 2. Load image to get dimensions
        img = Image.open(image_path)
        img_width, img_height = img.size
        
        # 3. Detect ingredients
        predictions = detect_ingredients(image_path)
        
        # 4. Validate detection
        is_valid, error_msg = validate_detection(predictions, confidence_threshold)
        if not is_valid:
            return handle_detection_error(image_path, error_msg)
        
        # 5. Get primary ingredient (highest confidence)
        primary = max(predictions['predictions'], key=lambda p: p['confidence'])
        
        # 6. Estimate size
        size_estimation = estimate_ingredient_size(
            primary, img_width, img_height, primary['class']
        )
        
        # 7. Create ingredient entity
        ingredient = create_ingredient_entity(primary, size_estimation, image_path)
        
        # 8. Return success result
        return {
            'success': True,
            'ingredient': ingredient,
            'all_detections': predictions['predictions'],
            'image_dimensions': {'width': img_width, 'height': img_height}
        }
        
    except Exception as e:
        return handle_detection_error(image_path, f"Processing error: {str(e)}")

print("✅ Complete recognition pipeline defined")

## 9. Visualization Function

In [ ]:
def visualize_detection(image_path, result):
    """
    Visualize detection results with bounding boxes
    
    Args:
        image_path: Path to image
        result: Recognition result dict
    """
    if not result['success']:
        print(f"❌ Detection failed: {result['error']}")
        return
    
    # Load image
    img = Image.open(image_path)
    fig, ax = plt.subplots(1, 1, figsize=(12, 8))
    ax.imshow(img)
    
    # Draw bounding boxes for all detections
    for detection in result['all_detections']:
        x = detection['x'] - detection['width'] / 2
        y = detection['y'] - detection['height'] / 2
        
        # Draw rectangle
        rect = patches.Rectangle(
            (x, y), detection['width'], detection['height'],
            linewidth=2, edgecolor='lime', facecolor='none'
        )
        ax.add_patch(rect)
        
        # Add label
        label = f"{detection['class']} ({detection['confidence']:.2f})"
        ax.text(x, y - 10, label, color='lime', fontsize=12,
                bbox=dict(boxstyle='round', facecolor='black', alpha=0.7))
    
    # Display primary ingredient info
    ingredient = result['ingredient']
    info_text = (
        f"Primary Ingredient: {ingredient['name']}\n"
        f"Confidence: {ingredient['confidence_score']:.2%}\n"
        f"Estimated Weight: {ingredient['estimated_quantity']['weight_grams']:.0f}g\n"
        f"Size Confidence: {ingredient['estimated_quantity']['confidence']:.2%}"
    )
    
    ax.text(0.02, 0.98, info_text, transform=ax.transAxes,
            fontsize=11, verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='white', alpha=0.9))
    
    ax.axis('off')
    plt.title('Ingredient Detection Result', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    # Print detailed info
    print("\n📊 Detection Summary:")
    print(f"  Primary: {ingredient['name']}")
    print(f"  Confidence: {ingredient['confidence_score']:.2%}")
    print(f"  Estimated Weight: {ingredient['estimated_quantity']['weight_grams']:.0f}g")
    print(f"  Total Detections: {len(result['all_detections'])}")

print("✅ Visualization function defined")

## 10. Test with Sample Image

**Note**: You need to place a test image in the `data/test_images/` folder to run this test.

In [ ]:
# Check for test images
test_images = list(DATA_TEST.glob("*.jpg")) + list(DATA_TEST.glob("*.png")) + list(DATA_TEST.glob("*.webp"))

if test_images:
    print(f"Found {len(test_images)} test images:")
    for img in test_images:
        print(f"  - {img.name}")
    
    # Test with first image
    test_image = test_images[0]
    print(f"\n🧪 Testing with: {test_image.name}")
    
    # Run recognition
    result = recognize_ingredient(test_image)
    
    # Visualize
    visualize_detection(test_image, result)
    
    # Save result
    if result['success']:
        result_file = RESULTS_DIR / f"{result['ingredient']['ingredient_id']}.json"
        with open(result_file, 'w') as f:
            json.dump(result, f, indent=2)
        print(f"\n✅ Result saved to: {result_file}")
else:
    print(f"⚠️ No test images found in {DATA_TEST}")
    print("\nPlease add test images (chicken breast, salmon, vegetables, etc.) to test the model")
    print("\nExample usage:")
    print("```python")
    print("result = recognize_ingredient('path/to/ingredient_photo.jpg')")
    print("visualize_detection('path/to/ingredient_photo.jpg', result)")
    print("```")

## 11. Summary

### ✅ Completed (Tasks T014-T017):
1. ✅ **T014**: Ingredient detection using Roboflow Inference SDK
2. ✅ **T015**: Size estimation from bounding box area
3. ✅ **T016**: Ingredient entity creation per data-model.md
4. ✅ **T017**: Validation and error handling

### 🎯 Key Features:
- **Serverless Inference**: Uses Roboflow API (no local model)
- **Object Detection**: Detects raw ingredients with bounding boxes
- **Confidence Scoring**: Filters low-confidence predictions (< 70%)
- **Size Estimation**: Estimates weight from bounding box area
- **Entity Creation**: Follows data-model.md schema
- **Error Handling**: User-friendly messages for failures

### 📊 Detection Capabilities:
- Raw meat (chicken, beef, pork, fish)
- Vegetables (carrot, broccoli, tomato, etc.)
- Fruits (apple, banana, orange, etc.)
- Other ingredients (eggs, cheese, etc.)

### 🔧 API Contract (ingredient_recognition.json):
- **Input**: Image file path
- **Output**: Ingredient entity with:
  - `ingredient_id` (UUID)
  - `name` (detected class)
  - `confidence_score` (≥0.7)
  - `estimated_quantity` (weight in grams)
  - `visual_characteristics` (bounding box)

### 💡 Advantages of Inference SDK:
- ✅ **Lightweight** - Minimal dependencies
- ✅ **Fast** - Direct HTTP calls
- ✅ **Simple** - One-line inference
- ✅ **No download** - Serverless deployment

### 📝 Next Steps:
1. Integrate with recipe generation pipeline (T018-T022)
2. Test with diverse ingredient photos
3. Fine-tune confidence thresholds if needed

In [ ]:
print("🎉 Ingredient recognition system ready!")
print(f"\n📁 Results directory: {RESULTS_DIR}")
print(f"🎯 Model: {MODEL_ID}")
print(f"📊 Confidence threshold: {CONFIDENCE_THRESHOLD}")
print("\n✅ Ready to detect ingredients from photos!")